# Chapter 3 — Tokenizers

Before a transformer sees any text it must be converted into numbers.
**Tokenization** is that conversion step: a raw string goes in, a sequence of
integer token IDs comes out. Different strategies make different trade-offs
between vocabulary size, out-of-vocabulary handling, and computational cost.

This chapter covers six strategies side-by-side, each applied to the same
sample sentences so the differences are directly comparable:

| Strategy | Library | Vocab required? |
|---|---|---|
| Whitespace | (built-in) | No |
| Regex | `re` | No |
| BPE | HuggingFace `tokenizers` | Trained |
| WordPiece | HuggingFace `tokenizers` | Trained |
| SentencePiece | `sentencepiece` | Trained |
| tiktoken | `tiktoken` | Pre-built (cl100k\_base) |


## Shared Corpus and Sample Sentence

The three learning-based tokenizers (BPE, WordPiece, SentencePiece) are trained
on the same small corpus. We then run all six tokenizers on a fixed **probe
sentence** so the outputs are directly comparable.


In [1]:
CORPUS = [
"the quick brown fox jumps over the lazy dog",
"transformers are powerful models for natural language processing",
"tokenization is the first step in any NLP pipeline",
"neural networks learn representations from data",
"attention mechanisms let every token query every other token",
"byte pair encoding merges the most frequent character pairs",
"word piece splits words into subword units using a trained vocabulary",
"sentence piece learns a subword vocabulary directly from raw text",
]

PROBE = "transformers tokenize text into subword units"
print('Corpus sentences:', len(CORPUS))
print('Probe:', PROBE)


Corpus sentences: 8
Probe: transformers tokenize text into subword units


## Whitespace Tokenizer

The simplest possible strategy: split on any whitespace. No vocabulary, no
training — just `str.split()`. Every word is a token, so the vocabulary is
unbounded and any unseen word becomes its own token. Useful for quick
prototyping or when the input is already clean.


In [2]:
from transformer_book_lab.whitespace_tokenizer import WhitespaceTokenizer

ws = WhitespaceTokenizer()
tokens = ws.tokenize(PROBE)
print('Tokens :', tokens)
print('Count  :', len(tokens))


Tokens : ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']
Count  : 6


## Regex Tokenizer

A step up from whitespace: use a **regular expression** to define what counts
as a token. The pattern `\w+` matches word characters (letters, digits,
underscores), which strips punctuation without needing a vocabulary. The caller
controls granularity by choosing the pattern.


In [3]:
from transformer_book_lab.regex_tokenizer import RegexTokenizer

rx = RegexTokenizer(pattern=r'\w+')
tokens = rx.tokenize(PROBE)
print('Pattern:', rx.pattern)
print('Tokens :', tokens)
print('Count  :', len(tokens))

# Demonstrate a finer-grained pattern that keeps apostrophes
rx2 = RegexTokenizer(pattern=r"[\w']+")
print("\nWith [\\w']+ on \"don't stop\":", rx2.tokenize("don't stop"))


Pattern: \w+
Tokens : ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']
Count  : 6

With [\w']+ on "don't stop": ["don't", 'stop']


## BPE Tokenizer (Byte-Pair Encoding)

BPE starts from individual characters (or bytes) and iteratively merges the
most frequent adjacent pair into a new token. Repeating this `vocab_size` times
produces a vocabulary that captures common subwords while keeping rare words
decomposable. GPT-2, RoBERTa, and most modern LLMs use BPE.

Our `BpeTokenizer` wraps HuggingFace `tokenizers` with a `ByteLevel`
pre-tokeniser so every byte is representable and the encode→decode round-trip
is lossless.


In [4]:
from transformer_book_lab.bpe_tokenizer import BpeTokenizer

bpe = BpeTokenizer(corpus=CORPUS, vocab_size=300)
ids    = bpe.encode(PROBE)
decoded = bpe.decode(ids)

print(f'Vocab size : {bpe.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 300
Token IDs  : [83, 265, 279, 69, 259, 76, 257, 82, 261, 280, 72, 89, 68, 261, 68, 87, 83, 220, 258, 83, 78, 273, 84, 292, 220, 84, 77, 72, 83, 82]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


## WordPiece Tokenizer

WordPiece is BERT's tokenizer. Like BPE it builds a subword vocabulary, but
instead of merging the *most frequent* pair it merges the pair that maximises
the likelihood of the training corpus under a unigram language model.

Continuation subwords are prefixed with `##` (e.g. `playing` → `play` + `##ing`),
making the word boundary recoverable at decode time.


In [5]:
from transformer_book_lab.wordpiece_tokenizer import WordPieceTokenizer

wp = WordPieceTokenizer(corpus=CORPUS, vocab_size=200)
ids     = wp.encode(PROBE)
decoded  = wp.decode(ids)

print(f'Vocab size : {wp.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 200
Token IDs  : [91, 103, 159, 167, 43, 199, 32, 151, 181, 84, 175, 116, 152, 119]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


## SentencePiece Tokenizer

SentencePiece treats the input as a raw stream of Unicode characters — spaces
included — so it is **language-agnostic** and works well for languages without
whitespace word boundaries (Chinese, Japanese). It supports both BPE and Unigram
model types; we use BPE here for comparability.

Unlike HuggingFace `tokenizers`, SentencePiece writes a model file during
training. Our wrapper keeps that inside a temporary directory so no files are
left on disk after construction.


In [6]:
from transformer_book_lab.sentencepiece_tokenizer import SentencePieceTokenizer

sp = SentencePieceTokenizer(corpus=CORPUS, vocab_size=50)
ids     = sp.encode(PROBE)
decoded  = sp.decode(ids)

print(f'Vocab size : {sp.vocab_size}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 50
Token IDs  : [3, 12, 24, 27, 36, 7, 34, 5, 27, 3, 26, 41, 4, 28, 45, 21, 3, 21, 44, 23, 20, 6, 23, 26, 14, 30, 40, 15, 33, 20, 30, 24, 28, 23, 27]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


## tiktoken Tokenizer (`cl100k_base`)

`tiktoken` is OpenAI's production tokenizer library. `cl100k_base` is the
encoding used by GPT-4, GPT-3.5-turbo, and the embeddings API. It has
**100 277 tokens** and uses a ByteLevel BPE trained on a vast multilingual
corpus. No training is needed here — the vocabulary is pre-built and cached
locally after the first download.

This is the most production-realistic tokenizer in this chapter.


In [7]:
from transformer_book_lab.tiktoken_tokenizer import TiktokenTokenizer

tt = TiktokenTokenizer()
ids     = tt.encode(PROBE)
decoded  = tt.decode(ids)

print(f'Vocab size : {tt.vocab_size:,}')
print(f'Token IDs  : {ids}')
print(f'Decoded    : {decoded!r}')
print(f'Round-trip : {decoded == PROBE}')


Vocab size : 100,277
Token IDs  : [4806, 388, 78751, 1495, 1139, 1207, 1178, 8316]
Decoded    : 'transformers tokenize text into subword units'
Round-trip : True


## Comparison: All Six Strategies on the Same Input

The table below shows how each tokenizer handles the probe sentence, making
the vocabulary size / token count trade-off visible.


In [8]:
rows = [
    ('Whitespace',    ws.tokenize(PROBE),   'unbounded',       '-'),
    ('Regex (\\w+)',   rx.tokenize(PROBE),   'unbounded',       '-'),
    ('BPE',           bpe.encode(PROBE),     bpe.vocab_size,    'trained'),
    ('WordPiece',     wp.encode(PROBE),      wp.vocab_size,     'trained'),
    ('SentencePiece', sp.encode(PROBE),       sp.vocab_size,     'trained'),
    ('tiktoken',      tt.encode(PROBE),       tt.vocab_size,     'pre-built'),
]

print(f'Probe: {PROBE!r}\n')
print(f'{"Strategy":<16} {"Tokens / IDs":<45} {"Count":>5}  {"Vocab":>8}  {"Training"}')
print('-' * 90)
for name, toks, vocab, training in rows:
    preview = str(toks[:6]) + ('...' if len(toks) > 6 else '')
    print(f'{name:<16} {preview:<45} {len(toks):>5}  {str(vocab):>8}  {training}')


Probe: 'transformers tokenize text into subword units'

Strategy         Tokens / IDs                                  Count     Vocab  Training
------------------------------------------------------------------------------------------
Whitespace       ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']     6  unbounded  -
Regex (\w+)      ['transformers', 'tokenize', 'text', 'into', 'subword', 'units']     6  unbounded  -
BPE              [83, 265, 279, 69, 259, 76]...                   30       300  trained
WordPiece        [91, 103, 159, 167, 43, 199]...                  14       200  trained
SentencePiece    [3, 12, 24, 27, 36, 7]...                        35        50  trained
tiktoken         [4806, 388, 78751, 1495, 1139, 1207]...           8    100277  pre-built


## What I Learned

- **Whitespace and regex tokenizers are stateless**: no training, no vocabulary.
  They work as long as word identity is preserved by whitespace, but they cannot
  share representations across morphological variants (`run`/`running`).
- **BPE, WordPiece, and SentencePiece are all subword strategies**: they
  decompose rare words into known pieces, bounding vocabulary size while keeping
  out-of-vocabulary tokens rare.
- **BPE merges by frequency; WordPiece merges by likelihood**: in practice the
  vocabularies look similar, but WordPiece's `##` prefix makes word boundaries
  explicit in the token stream.
- **SentencePiece treats spaces as characters**: this makes it language-agnostic
  and lets it learn tokenization from any raw text without a pre-tokenizer.
- **tiktoken is a fixed, production-grade BPE vocabulary**: no training required,
  100 277 tokens covering essentially all Unicode, deterministic across versions.
- **Vocab size trades off against token count**: a larger vocabulary encodes text
  in fewer tokens (shorter sequences = faster attention), but requires more
  embedding parameters.


## Final Exercise

Train a `BpeTokenizer` and a `WordPieceTokenizer` on the shared `CORPUS` with
`vocab_size=150`. Encode the probe sentence with both and print:

1. The token count for each.
2. Whether the two tokenizers produce the **same** token IDs for the probe.

**Why this matters**: BPE and WordPiece use the same training data and target
vocabulary size but different merge criteria. Comparing their outputs on the same
input reveals whether the criteria matter in practice for small corpora.


In [9]:
from transformer_book_lab.bpe_tokenizer import BpeTokenizer
from transformer_book_lab.wordpiece_tokenizer import WordPieceTokenizer

bpe150 = BpeTokenizer(corpus=CORPUS, vocab_size=150)
wp150  = WordPieceTokenizer(corpus=CORPUS, vocab_size=150)

bpe_ids = bpe150.encode(PROBE)
wp_ids  = wp150.encode(PROBE)

print(f'BPE token count      : {len(bpe_ids)}')
print(f'WordPiece token count: {len(wp_ids)}')
print(f'Same IDs?            : {bpe_ids == wp_ids}')


BPE token count      : 45
WordPiece token count: 22
Same IDs?            : False


### Conclusion

With `vocab_size=150` on the same small corpus, BPE produced **45 tokens** for
the probe sentence while WordPiece produced only **22**,roughly half. The IDs
were also completely different (`Same IDs? False`), for two distinct reasons:

**Why the token counts differ so much.**
WordPiece selects merges that maximise corpus likelihood rather than raw
frequency. On a small corpus this tends to produce longer, more linguistically
coherent subwords earlier in the merge sequence. BPE, by contrast, merges
whatever pair appears most often, on a tiny corpus that often means short
character n-grams, leaving more words fragmented at the same vocab budget.
In other words, the merge criterion matters, especially when training data is
scarce.

**Why the IDs are always different.**
Each tokenizer builds its own vocabulary index independently, so even if two
tokenizers produced the *same* segmentation (e.g., both split `transformers`
into `transform` + `##ers`), the integer IDs assigned to those pieces would
still differ. ID equality is only meaningful *within* one tokenizer's
encode/decode cycle.

**Takeaway for transformer design.**
Choosing BPE vs WordPiece is not just a detail, it affects sequence length,
which directly drives attention cost (O(n²)). On small or domain-specific
corpora, WordPiece's likelihood-based criterion can be noticeably more
efficient than BPE's frequency-based one at the same vocabulary size.
